In [33]:
import pandas as pd

In [34]:
filings = pd.read_parquet("data/filings.parquet")

In [35]:
filings.shape

(73, 6)

In [36]:
filings.head()

,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
1,2,AAPL,2024-05-02,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
2,3,AAPL,2024-08-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
3,4,AAPL,2024-11-01,10-K,Item 7. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
4,5,AAPL,2025-01-31,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...


In [37]:
filings.columns

Index(['id', 'symbol', 'date', 'document_type', 'mda_content', 'risk_content'], dtype='str')

In [38]:
filings.dtypes

id                int32
symbol              str
date             object
document_type       str
mda_content         str
risk_content        str
dtype: object

In [39]:
filings.isna().sum()

id               0
symbol           0
date             0
document_type    0
mda_content      0
risk_content     0
dtype: int64

In [40]:
filings.duplicated().sum()

np.int64(0)

In [41]:
id_counts = filings["id"].value_counts()
repeated_ids = id_counts[id_counts > 1]

print("Unique ID count:", filings["id"].nunique())
print("Are IDs unique?:", filings["id"].is_unique)
print("Number of repeated ID values:", len(repeated_ids))
print("Number of rows with repeated IDs:", int(repeated_ids.sum()))
print(repeated_ids)

Unique ID count: 73
Are IDs unique?: True
Number of repeated ID values: 0
Number of rows with repeated IDs: 0
Series([], Name: count, dtype: int64)


In [42]:
date_temp = pd.to_datetime(filings["date"], errors="coerce")
failed_date_mask = filings["date"].notna() & date_temp.isna()

print("Failed datetime conversions:", int(failed_date_mask.sum()))
print(filings.loc[failed_date_mask, ["id", "date"]])

Failed datetime conversions: 0
Empty DataFrame
Columns: [id, date]
Index: []


In [43]:
document_type_counts = filings["document_type"].value_counts(dropna=False)
print(document_type_counts)

document_type
10-Q    51
10-K    22
Name: count, dtype: int64


In [44]:
symbol_counts = filings["symbol"].value_counts(dropna=False)
print(symbol_counts)

symbol
ADBE     10
AAPL      9
AMZN      9
GOOGL     9
META      9
MSFT      9
NVDA      9
TSLA      9
Name: count, dtype: int64


In [45]:
symbol_whitespace = filings["symbol"].astype(str).str.contains(r"^\s+|\s+$", na=False)
symbol_case_variants = filings["symbol"].astype(str).str.strip().ne(filings["symbol"].astype(str).str.strip().str.upper())
document_type_whitespace = filings["document_type"].astype(str).str.contains(r"^\s+|\s+$", na=False)
document_type_case_variants = filings["document_type"].astype(str).str.strip().ne(filings["document_type"].astype(str).str.strip().str.upper())

print("Leading/trailing whitespace in symbol:", int(symbol_whitespace.sum()))
print("Mixed-case symbol values:", int(symbol_case_variants.sum()))
print(filings.loc[symbol_whitespace | symbol_case_variants, ["id", "symbol"]])
print("Leading/trailing whitespace in document_type:", int(document_type_whitespace.sum()))
print("Mixed-case document_type values:", int(document_type_case_variants.sum()))
print(filings.loc[document_type_whitespace | document_type_case_variants, ["id", "document_type"]])

Leading/trailing whitespace in symbol: 0
Mixed-case symbol values: 0
Empty DataFrame
Columns: [id, symbol]
Index: []
Leading/trailing whitespace in document_type: 0
Mixed-case document_type values: 0
Empty DataFrame
Columns: [id, document_type]
Index: []


In [46]:
mda_blank = filings["mda_content"].fillna("").astype(str).str.strip().eq("")
risk_blank = filings["risk_content"].fillna("").astype(str).str.strip().eq("")

print("Blank mda_content rows:", int(mda_blank.sum()))
print(filings.loc[mda_blank, ["id", "symbol", "document_type", "mda_content"]])
print("Blank risk_content rows:", int(risk_blank.sum()))
print(filings.loc[risk_blank, ["id", "symbol", "document_type", "risk_content"]])

Blank mda_content rows: 0
Empty DataFrame
Columns: [id, symbol, document_type, mda_content]
Index: []
Blank risk_content rows: 0
Empty DataFrame
Columns: [id, symbol, document_type, risk_content]
Index: []


In [47]:
date_temp = pd.to_datetime(filings["date"], errors="coerce")
print("Minimum date:", date_temp.min())
print("Maximum date:", date_temp.max())

Minimum date: 2024-01-16 00:00:00
Maximum date: 2026-03-25 00:00:00


In [48]:
date_failed = pd.to_datetime(filings["date"], errors="coerce").isna() & filings["date"].notna()
suspicious_mask = (
    date_failed
    | filings["symbol"].astype(str).str.contains(r"^\s+|\s+$", na=False)
    | filings["document_type"].astype(str).str.contains(r"^\s+|\s+$", na=False)
    | filings["mda_content"].fillna("").astype(str).str.strip().eq("")
    | filings["risk_content"].fillna("").astype(str).str.strip().eq("")
)

print("Suspicious records to review:", int(suspicious_mask.sum()))
print(filings.loc[suspicious_mask, ["id", "symbol", "date", "document_type", "mda_content", "risk_content"]])

Suspicious records to review: 0
Empty DataFrame
Columns: [id, symbol, date, document_type, mda_content, risk_content]
Index: []


## Cleaning decision

The only change made in this assignment was converting the `date` field to pandas datetime. The value was stored as an object/string even though it represents dates, so this conversion makes the field usable for time-based analysis without deleting any rows. All other columns were left unchanged because the inspection showed no missing values, no exact duplicates, no repeated IDs, no blank text issues, and no obvious text-standardization problems.

In [49]:
date_rows = len(filings)
missing_dates = int(filings["date"].isna().sum())
date_temp = pd.to_datetime(filings["date"], errors="coerce")
failed_date_conversions = int((filings["date"].notna() & date_temp.isna()).sum())
print("Number of rows:", date_rows)
print("Number of missing dates:", missing_dates)
print("Number of failed date conversions:", failed_date_conversions)
print("Minimum date:", date_temp.min())
print("Maximum date:", date_temp.max())

Number of rows: 73
Number of missing dates: 0
Number of failed date conversions: 0
Minimum date: 2024-01-16 00:00:00
Maximum date: 2026-03-25 00:00:00


In [50]:
filings_clean = filings.copy()
filings_clean["date"] = pd.to_datetime(filings_clean["date"], errors="coerce")

In [51]:
filings_clean.shape

(73, 6)

In [52]:
filings_clean.dtypes

id                       int32
symbol                     str
date             datetime64[s]
document_type              str
mda_content                str
risk_content               str
dtype: object

In [53]:
filings_clean.head()

,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
1,2,AAPL,2024-05-02,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
2,3,AAPL,2024-08-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
3,4,AAPL,2024-11-01,10-K,Item 7. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
4,5,AAPL,2025-01-31,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...


In [54]:
filings_clean.isna().sum()

id               0
symbol           0
date             0
document_type    0
mda_content      0
risk_content     0
dtype: int64

In [55]:
filings_clean.duplicated().sum()

np.int64(0)

## 1) Row identity and unique IDs

One row represents one SEC filing record. The `id` column should uniquely identify each row if the dataset is clean and consistent.

In [56]:
print("Rows:", len(filings))
print("Unique IDs:", filings["id"].nunique())
print("Are IDs unique?", filings["id"].is_unique)
print("Rows with repeated IDs:", int(filings["id"].duplicated().sum()))

Rows: 73
Unique IDs: 73
Are IDs unique? True
Rows with repeated IDs: 0


## 2) Cleaning decision log

This log records the single allowed change and documents why other checks did not require changes.

In [57]:
decision_log = pd.DataFrame({
    "rule": [
        "Convert date to datetime with errors='coerce'",
        "Keep all rows",
        "Do not remove missing values",
        "Do not remove duplicates",
        "Do not normalize text fields"
    ],
    "reason": [
        "The date field is stored as an object/string even though it represents dates and should be usable for time-based analysis.",
        "The inspection found no rows that needed to be dropped because there were no missing values or obvious invalid records.",
        "There were 0 missing values across all columns, so removing rows would not improve data quality.",
        "There were 0 exact duplicate rows and 0 repeated IDs, so dropping rows would not be justified.",
        "The inspection found 0 whitespace issues, 0 mixed-case inconsistencies, and 0 blank text fields in the text columns, so no normalization was necessary."
    ],
    "check": [
        "pd.to_datetime(filings['date'], errors='coerce')",
        "len(filings_clean) == len(filings)",
        "filings.isna().sum().sum() == 0",
        "filings.duplicated().sum() == 0 and filings['id'].is_unique",
        "symbol/document_type whitespace and case checks were all 0; mda_content and risk_content blank counts were 0"
    ]
})

display(decision_log)

,rule,reason,check
0,Convert date to datetime with errors='coerce',The date field is stored as an object/string e...,"pd.to_datetime(filings['date'], errors='coerce')"
1,Keep all rows,The inspection found no rows that needed to be...,len(filings_clean) == len(filings)
2,Do not remove missing values,There were 0 missing values across all columns...,filings.isna().sum().sum() == 0
3,Do not remove duplicates,There were 0 exact duplicate rows and 0 repeat...,filings.duplicated().sum() == 0 and filings['i...
4,Do not normalize text fields,"The inspection found 0 whitespace issues, 0 mi...",symbol/document_type whitespace and case check...


## 3) Original vs. cleaned row counts

The date conversion should not change the number of rows.

In [58]:
original_rows = len(filings)
cleaned_rows = len(filings_clean)
print("Original rows:", original_rows)
print("Cleaned rows:", cleaned_rows)
print("Difference:", cleaned_rows - original_rows)

Original rows: 73
Cleaned rows: 73
Difference: 0


## 4) Verify unchanged text and ID fields

The date conversion should not change the values in the other columns.

In [59]:
checks = {
    "symbol_equal": filings["symbol"].equals(filings_clean["symbol"]),
    "document_type_equal": filings["document_type"].equals(filings_clean["document_type"]),
    "mda_content_equal": filings["mda_content"].equals(filings_clean["mda_content"]),
    "risk_content_equal": filings["risk_content"].equals(filings_clean["risk_content"]),
    "id_equal": filings["id"].equals(filings_clean["id"])
}

for name, result in checks.items():
    print(name + ":", result)

symbol_equal: True
document_type_equal: True
mda_content_equal: True
risk_content_equal: True
id_equal: True


## 5) Document type and symbol coverage

The cleaned data should still contain the same categories and same companies as the original.

In [60]:
document_type_counts = filings_clean["document_type"].value_counts().sort_index()
print(document_type_counts)
print("Unique company symbols:", filings_clean["symbol"].nunique())

document_type
10-K    22
10-Q    51
Name: count, dtype: int64
Unique company symbols: 8


## 6) Impossible or suspicious dates

Check for any date conversion failures and confirm the valid date range.

In [61]:
cleaned_dates = pd.to_datetime(filings_clean["date"], errors="coerce")
failed_date_conversions = int((filings_clean["date"].notna() & cleaned_dates.isna()).sum())
suspicious_date_rows = filings_clean.loc[cleaned_dates.isna(), ["id", "symbol", "date"]]

print("Failed date conversions:", failed_date_conversions)
print("Suspicious date rows:")
display(suspicious_date_rows)
print("Minimum date:", cleaned_dates.min())
print("Maximum date:", cleaned_dates.max())

Failed date conversions: 0
Suspicious date rows:


,id,symbol,date


Minimum date: 2024-01-16 00:00:00
Maximum date: 2026-03-25 00:00:00


## 7) Five record checks for date conversion evidence

These five records provide coverage across different document types and dates.

In [62]:
check_ids = [1, 2, 3, 4, 5]
original_check = filings.loc[filings["id"].isin(check_ids), ["id", "date", "document_type"]].copy()
original_check = original_check.rename(columns={"date": "original_date"})
original_check["expected_cleaned_date"] = pd.to_datetime(original_check["original_date"], errors="coerce")

actual_check = filings_clean.loc[filings_clean["id"].isin(check_ids), ["id", "date", "document_type"]].copy()
actual_check = actual_check.rename(columns={"date": "actual_cleaned_date"})

record_checks = original_check.merge(actual_check, on=["id", "document_type"], how="left")
record_checks["expected_equals_actual"] = record_checks["expected_cleaned_date"] == record_checks["actual_cleaned_date"]

record_checks = record_checks[["id", "original_date", "expected_cleaned_date", "actual_cleaned_date", "expected_equals_actual"]]
display(record_checks)

,id,original_date,expected_cleaned_date,actual_cleaned_date,expected_equals_actual
0,1,2024-02-01,2024-02-01,2024-02-01,True
1,2,2024-05-02,2024-05-02,2024-05-02,True
2,3,2024-08-01,2024-08-01,2024-08-01,True
3,4,2024-11-01,2024-11-01,2024-11-01,True
4,5,2025-01-31,2025-01-31,2025-01-31,True


## 8) Cleaned sample for review

This sample keeps the five checked records and adds a reasonable selection of other filing rows, while excluding unnecessary full text output.

In [63]:
checked_ids = [1, 2, 3, 4, 5]
other_ids = filings_clean.loc[~filings_clean["id"].isin(checked_ids), "id"].tolist()
selected_ids = checked_ids + other_ids[:45]
filings_clean_sample = filings_clean[filings_clean["id"].isin(selected_ids)][["id", "symbol", "date", "document_type"]].copy()

print("Sample rows:", len(filings_clean_sample))
display(filings_clean_sample.head(10))

Sample rows: 50


,id,symbol,date,document_type
0,1,AAPL,2024-02-01,10-Q
1,2,AAPL,2024-05-02,10-Q
2,3,AAPL,2024-08-01,10-Q
3,4,AAPL,2024-11-01,10-K
4,5,AAPL,2025-01-31,10-Q
5,6,AAPL,2025-05-02,10-Q
6,7,AAPL,2025-08-01,10-Q
7,8,AAPL,2025-10-31,10-K
8,9,AAPL,2026-01-30,10-Q
9,10,ADBE,2024-01-16,10-K


## 9) Final verification summary

This summary combines the starting state, the final cleaned state, and key validity checks.

In [64]:
final_summary = pd.DataFrame({
    "metric": [
        "starting_rows",
        "final_rows",
        "missing_values",
        "duplicate_rows",
        "failed_date_conversions",
        "ids_remain_unique"
    ],
    "value": [
        len(filings),
        len(filings_clean),
        int(filings_clean.isna().sum().sum()),
        int(filings_clean.duplicated().sum()),
        int(filings_clean["date"].isna().sum()),
        bool(filings_clean["id"].is_unique)
    ]
})

display(final_summary)

,metric,value
0,starting_rows,73
1,final_rows,73
2,missing_values,0
3,duplicate_rows,0
4,failed_date_conversions,0
5,ids_remain_unique,True
